![Platform+](images/banner.png)

<div style="max-width:150px; float:right;">

![The Roastery Coffee Co.](images/roastery-logo.png)

</div>

# Lab 10: Capstone — Roastery Intelligence
## The VP of Operations flagged a revenue dip at one store. Find out why — and leave something behind.

> **Persona track: Data Analyst** — same mission, your angle.

:point_right: **The VP just pinged you: _"Roastery Mission's numbers look off. Figure out what happened, and give me something the team can re-run every month without bugging you."_**

This is a capstone — you'll pull together skills from every lab in the course to run one real investigation end to end, then package the answer as a reusable view the ops team owns.

📚 **By the end you'll have:**
- Diagnosed an access problem and fixed it with RBAC
- Built a rolling-window revenue comparison and charted it
- Verified a handed-off dataset before trusting it
- Loaded external CSV data from a stage into Snowflake
- Enriched the story with Marketplace data and AI-generated pairings
- Scored and classified customer feedback with AI functions
- Parsed an unstructured PDF to cross-reference complaints
- Recovered from an accidental delete using Time Travel
- Packaged everything into a reusable view for the ops team

**Time:** ~45 min · **Lab interface:** Cortex Code (CoCo) is your tutor — ask it to explain or fix anything.
*Source data lives in the shared, read-only `PLATPLUS_DB`; everything you create lives in your own database, warehouse, and role — each named after your username (e.g. `USER_DB` / `USER_WH` / `USER_RL`).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

> 💡 **Coming from another lab?** Shut down that notebook's kernel first — click **Connected** at the top → **Shut down kernel** in the previous notebook. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for this capstone.

> :mortar_board: **This is a capstone — not a tutorial.** Every section calls back to a skill from an earlier lab, and most cells are yours to write. Each section gives you **three paths to the answer (your choice)** — you can try all three if you wish, but you only need one:
>
> 1. **Write it yourself** in the empty cells (the real challenge — prove you learned it)
> 2. **Reveal the 🧩 Stuck? Reference solution** and paste it in (see the answer directly)
> 3. **Use the 🤖 CoCo assist prompt** to have CoCo generate the code for you (practice the AI-assisted workflow)
>
> All three paths produce the same result going into the same empty cells. Pick whichever level of help you need — there's no penalty for using the hints.

> :rotating_light: **IMPORTANT — CoCo vs notebook SQL cells:** As you go through this lab and use the CoCo assist prompts, be aware that curly braces `{}` in SQL (e.g. `{'mode': 'LAYOUT'}`) can conflict with CoCo's internal SQL execution. If CoCo reports an error but the SQL looks correct, just copy it into a notebook SQL cell and run it there — it will work fine. This is a difference in how CoCo executes SQL internally vs how notebook cells send it directly to Snowflake.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)


## Set your session context
Every query runs under a role (what you're allowed to do) and a warehouse (compute). The cell below switches to your own role, creates and uses your own warehouse, and creates a `SANDBOX` schema for what you'll build in this lab.

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.SANDBOX;
USE DATABASE {{my_db}};
USE SCHEMA SANDBOX;


---
## 1 · Access Gate (Lab 03 — RBAC)

Before you investigate anything, you need access. Start by switching to your restricted analyst role — **with secondary roles off** — and try to query the feedback table. You'll hit a wall, and clearing it is the first challenge.

**Your task:**
1. Switch to `{{my_analyst_role}}` AND disable secondary roles so it's truly isolated
2. Try: `SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK LIMIT 5;`
3. You'll get "does not exist or not authorized." Diagnose it with `SHOW GRANTS TO ROLE {{my_analyst_role}};`
4. Switch back to `{{my_role}}` and re-enable secondary roles, then re-run the query

> 📌 **Recall Lab 03:** `USE SECONDARY ROLES ALL` combines privileges from all your granted roles — so even a restricted role can "see" everything if secondary roles are on. Turn them off with `USE SECONDARY ROLES NONE` to test a role in isolation. This is how you confirm what a role can *really* do on its own.

> 🤖 **CoCo assist** *(try the empty cells yourself first — if stuck, reveal the 🧩 solution below OR use this prompt):*
>
> *"I got 'does not exist or not authorized' when querying PLATPLUS_DB.ROASTERY_SALES.FEEDBACK. How do I find out which role has access?"*
>
> These prompts are starting points — you may need to iterate. Refine your question based on CoCo's response, just like you would with a colleague.

> 📌 **Why this reset?** Before testing what a role *can't* do, you need a clean baseline. If a previous lab granted access and never revoked it, or if secondary roles are leaking privileges from other roles, the RBAC test gives a false positive. This is real-world hygiene: **reset state → isolate the role → test → verify.** (The revokes are no-ops if the grants don't exist — safe to run either way.)

In [ ]:
-- Reset: ensure the restricted role has NO access to PLATPLUS_DB feedback
-- (These are no-ops if the grants don't exist — safe to run regardless)
USE ROLE {{my_role}};
REVOKE SELECT ON TABLE PLATPLUS_DB.ROASTERY_SALES.FEEDBACK FROM ROLE {{my_analyst_role}};
REVOKE USAGE ON SCHEMA PLATPLUS_DB.ROASTERY_SALES FROM ROLE {{my_analyst_role}};
REVOKE USAGE ON DATABASE PLATPLUS_DB FROM ROLE {{my_analyst_role}};


In [ ]:
-- Step 1: Switch to your restricted analyst role with secondary roles OFF


In [ ]:
-- Step 2: Try to query feedback (expect an error under the isolated restricted role!)


In [ ]:
-- Step 3: Diagnose — what can your restricted role see on its own?


In [ ]:
-- Step 4: Fix — re-enable secondary roles


In [ ]:
-- IMPORTANT: Run this after completing Section 1, regardless of which path you used above
-- Restore: switch back to your primary role and re-enable secondary roles for the rest of the lab
USE ROLE {{my_role}};
USE SECONDARY ROLES ALL;


> <details><summary>🧩 Stuck? Reference solution</summary>
>
> ```sql
> -- Step 1: Switch to your restricted analyst role with secondary roles OFF
> USE ROLE {{my_analyst_role}};
> USE SECONDARY ROLES NONE;
>
> -- Step 2: Try to query feedback (expect an error under the isolated restricted role!)
> SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK LIMIT 5;
>
> -- Step 3: Diagnose — what can your restricted role see on its own?
> SHOW GRANTS TO ROLE {{my_analyst_role}};
> -- Notice: no grants on PLATPLUS_DB at all
>
> -- Step 4: Fix — switch back to your primary role and re-enable secondary roles
> USE ROLE {{my_role}};
> USE SECONDARY ROLES ALL;
> SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK LIMIT 5;
> ```
>
> **Why it failed:** `{{my_analyst_role}}` has no grants on `PLATPLUS_DB`. The fix is switching to `{{my_role}}`, which has a direct `SELECT` grant on `FEEDBACK`. We used `USE SECONDARY ROLES NONE` at the start to isolate the restricted role — otherwise secondary roles leak privileges from your other roles and mask the problem.
>
> </details>

---
## 2 · Investigate the Dip (Labs 01 & 02 — Navigate + Verify)

Now that you have access, build the revenue comparison. A **rolling window** (trailing 30 days vs. the prior 30 days) avoids the calendar-month trap that can smear or hide a dip.

**Your task:** Write a query that joins `PLATPLUS_DB.ROASTERY_SALES.ORDERS` to `PLATPLUS_DB.ROASTERY_SALES.STORES` and computes:
- `sales_last_30d` — revenue in the most recent 30 days
- `sales_prior_30d` — revenue in the 30 days before that
- `pct_change` — the percentage difference

Then plot it as a Plotly bar chart so the dip store jumps out visually. The query below is filled in as your anchor example — run it, read the result, then build the chart yourself.

> 📌 **Recall Lab 02:** A rolling window compares the most recent 30 days to the prior 30. This training dataset is a fixed snapshot (orders stop on a set date), so we anchor the window to the **latest order date** — `MAX(order_ts)` — instead of `CURRENT_TIMESTAMP()`. That keeps the comparison meaningful no matter when you run the lab.

> 🤖 **CoCo assist (chart)** *(try the empty cells yourself first — if stuck, reveal the 🧩 solution below OR use this prompt):*
>
> *"Using the dip_analysis SQL cell below as the query, create a Plotly bar chart in the dip_chart Python cell with STORE_NAME on the x-axis and PCT_CHANGE on the y-axis."*

In [ ]:
-- Rolling-window revenue comparison: last 30 days vs prior 30 days, per store
-- Anchored to the latest order date (static training snapshot) instead of CURRENT_TIMESTAMP()
WITH latest AS (
    SELECT MAX(order_ts) AS max_ts FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS
)
SELECT s.store_name,
       SUM(CASE WHEN o.order_ts >= DATEADD(day, -30, l.max_ts)
           THEN o.total_amount ELSE 0 END) AS sales_last_30d,
       SUM(CASE WHEN o.order_ts >= DATEADD(day, -60, l.max_ts)
                 AND o.order_ts < DATEADD(day, -30, l.max_ts)
           THEN o.total_amount ELSE 0 END) AS sales_prior_30d,
       ROUND(100.0 * (sales_last_30d - sales_prior_30d) / NULLIF(sales_prior_30d, 0), 1) AS pct_change
FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS o
JOIN PLATPLUS_DB.ROASTERY_SALES.STORES s ON s.store_id = o.store_id
CROSS JOIN latest l
WHERE o.order_ts >= DATEADD(day, -60, l.max_ts)
GROUP BY s.store_name
ORDER BY pct_change;


In [ ]:
# YOUR TURN: Paste CoCo's chart code on the blank line below (after the import) and run it.
# The data comes from the dip_analysis SQL cell above — use session.sql("<that query>").to_pandas()
import plotly.express as px


> <details><summary>🧩 Stuck? Reference solution</summary>
>
> **Paste into `dip_chart` cell:**
>
> ```python
> import plotly.express as px
> max_ts = session.sql("SELECT MAX(order_ts)::DATE FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS").collect()[0][0]
> df = session.sql("""
>     WITH latest AS (SELECT MAX(order_ts) AS max_ts FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS)
>     SELECT s.store_name AS STORE_NAME,
>            ROUND(100.0 * (SUM(CASE WHEN o.order_ts >= DATEADD(day,-30,l.max_ts) THEN o.total_amount ELSE 0 END)
>                          - SUM(CASE WHEN o.order_ts >= DATEADD(day,-60,l.max_ts)
>                                      AND o.order_ts < DATEADD(day,-30,l.max_ts) THEN o.total_amount ELSE 0 END))
>                  / NULLIF(SUM(CASE WHEN o.order_ts >= DATEADD(day,-60,l.max_ts)
>                                     AND o.order_ts < DATEADD(day,-30,l.max_ts) THEN o.total_amount ELSE 0 END), 0), 1) AS PCT_CHANGE
>     FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS o
>     JOIN PLATPLUS_DB.ROASTERY_SALES.STORES s ON s.store_id = o.store_id
>     CROSS JOIN latest l
>     WHERE o.order_ts >= DATEADD(day,-60,l.max_ts)
>     GROUP BY s.store_name
>     ORDER BY PCT_CHANGE
> """).to_pandas()
> fig = px.bar(df, x="STORE_NAME", y="PCT_CHANGE",
>              title=f"Revenue % Change: Last 30 Days vs Prior 30 Days (as of {max_ts})",
>              color_discrete_sequence=["#29B5E8"])
> fig.update_layout(xaxis_title="Store", yaxis_title="% change")
> fig
> ```
>
> One bar drops well below the rest — that's your dip store. The `session.sql(...).to_pandas()` pattern pulls a query result straight into pandas for charting.
>
> </details>

### 📦 Before you build on it — verify the product handoff (Lab 02)

In a real workplace, a Data Engineering team would publish curated views for analysts to consume. In this lab, that's already been done for you — `PLATPLUS_DB.ROASTERY_INTELLIGENCE.V_PRODUCTS_ENRICHED` is a pre-built standardized product catalog with an `ENRICHED_AT` freshness timestamp. Before you lean on it, **check it.** Two rows are stale (their enrichment is over a month old) while everything else was refreshed recently — exactly the "verify before you trust" lesson from Lab 02, applied to a handoff instead of a query result.

> 📌 **You don't need to run any other lab first.** This view is already provisioned in the shared `PLATPLUS_DB` database. Your job here is what any analyst should do: verify the handoff before building on it.

**Your task:** peek at the handoff's freshness, then build a filtered view that excludes anything not refreshed in the last week. The two cells below are **already filled in as your anchor** — run them, read the output, and note how the canary confirms the two stale rows are gone.

In [ ]:
-- Peek at the handoff — how fresh is each row?
SELECT product_id, product_name, enriched_at,
       DATEDIFF(day, enriched_at, CURRENT_TIMESTAMP()) AS age_days
FROM PLATPLUS_DB.ROASTERY_INTELLIGENCE.V_PRODUCTS_ENRICHED
ORDER BY enriched_at;


In [ ]:
-- Exclude anything not refreshed in the last week before you build on it
CREATE OR REPLACE VIEW {{my_db}}.SANDBOX.V_PRODUCTS_FRESH AS
SELECT * FROM PLATPLUS_DB.ROASTERY_INTELLIGENCE.V_PRODUCTS_ENRICHED
WHERE enriched_at >= DATEADD(day, -7, CURRENT_TIMESTAMP());

-- Canary: confirm the stale products are gone (expect ZERO rows)
SELECT product_id, product_name FROM {{my_db}}.SANDBOX.V_PRODUCTS_FRESH
WHERE product_id IN (103, 106);


> ✅ **Zero rows.** `Cold Brew` (103) and `Blueberry Muffin` (106) — the stale rows — are excluded. `{{my_db}}.SANDBOX.V_PRODUCTS_FRESH` is what you'd reference from here on, not the raw handoff. A stale row hiding in an otherwise-fresh table is easy to miss if you never check the freshness column.

---
## 3 · Load the Promo Data (Lab 07 — Getting Data In)

The ops team sent a CSV of planned promotions. You need to bring it into Snowflake to check which stores actually ran their promos — and whether that explains the dip.

**Your task:**
1. Create a stage in `{{my_db}}.SANDBOX` (provided)
2. Create a CSV file format — comma-delimited, has a header row (provided)
3. Upload the CSV to the stage (a Python cell is provided — just run it)
4. Create a table and `COPY INTO` it
5. Query: which store(s) had their promos **cancelled**?

> 📌 **Recall Lab 07:** The pattern is always **stage → file format → COPY INTO**. `SKIP_HEADER=1` because this CSV has a header row. A bad row makes the whole load abort by default (`ON_ERROR = ABORT_STATEMENT`).

> 🤖 **CoCo assist** *(try the empty cells yourself first — if stuck, reveal the 🧩 solution below OR use this prompt):*
>
> *"Help me load a CSV called promotions.csv from a stage into a table. The CSV has columns: store_name, promo_name, start_date, end_date, discount_pct, status. It has a header row."*

In [ ]:
-- Create a stage and file format for the promo CSV
CREATE STAGE IF NOT EXISTS {{my_db}}.SANDBOX.CAPSTONE_STAGE;

CREATE OR REPLACE FILE FORMAT {{my_db}}.SANDBOX.CAPSTONE_CSV
  TYPE = 'CSV'
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  SKIP_HEADER = 1;


The cell below writes the CSV to a temp file and PUTs it to your stage. In the real world this file might land via email, SFTP, or a shared drive — the point is: **data outside Snowflake typically needs to be staged and loaded before you can query it with SQL.** Just run it.

In [ ]:
# PUT the CSV to your stage: write it to a temp file the container can access, then upload
csv_content = """store_name,promo_name,start_date,end_date,discount_pct,status
Roastery Downtown,Summer Boost,2026-06-15,2026-07-15,15,completed
Roastery Pike,Summer Boost,2026-06-15,2026-07-15,15,completed
Roastery Mission,Summer Boost,2026-06-15,2026-07-15,15,cancelled
Roastery Loop,Summer Boost,2026-06-15,2026-07-15,15,completed
Roastery Midtown,Summer Boost,2026-06-15,2026-07-15,15,completed
Roastery Harbor,Summer Boost,2026-06-15,2026-07-15,15,completed
Roastery Downtown,Loyalty Double Points,2026-07-01,2026-07-31,0,active
Roastery Pike,Loyalty Double Points,2026-07-01,2026-07-31,0,active
Roastery Mission,Loyalty Double Points,2026-07-01,2026-07-31,0,cancelled
Roastery Loop,Loyalty Double Points,2026-07-01,2026-07-31,0,active
Roastery Midtown,Loyalty Double Points,2026-07-01,2026-07-31,0,active
Roastery Harbor,Loyalty Double Points,2026-07-01,2026-07-31,0,active
"""

tmp_path = '/tmp/promotions.csv'
with open(tmp_path, 'w') as f:
    f.write(csv_content)

session.file.put(tmp_path, f'@{my_db}.SANDBOX.CAPSTONE_STAGE', auto_compress=False, overwrite=True)
print('File uploaded to stage.')


In [ ]:
-- Create the PROMOTIONS table and COPY INTO it from the stage


In [ ]:
-- Which store(s) had cancelled promotions?


> <details><summary>🧩 Stuck? Reference solution</summary>
>
> **Paste into `promo_copy` cell:**
>
> ```sql
> -- Create the PROMOTIONS table and COPY INTO it from the stage
> CREATE OR REPLACE TABLE {{my_db}}.SANDBOX.PROMOTIONS (
>   store_name VARCHAR,
>   promo_name VARCHAR,
>   start_date DATE,
>   end_date DATE,
>   discount_pct NUMBER,
>   status VARCHAR
> );
>
> COPY INTO {{my_db}}.SANDBOX.PROMOTIONS
> FROM @{{my_db}}.SANDBOX.CAPSTONE_STAGE/promotions.csv
> FILE_FORMAT = (FORMAT_NAME = '{{my_db}}.SANDBOX.CAPSTONE_CSV');
> ```
>
> **Paste into `promo_cancelled` cell:**
>
> ```sql
> -- Which store(s) had cancelled promotions?
> SELECT * FROM {{my_db}}.SANDBOX.PROMOTIONS
> WHERE status = 'cancelled';
> -- Result: Roastery Mission had BOTH promos cancelled.
> ```
>
> The `COPY INTO` reads the file you PUT to the stage, parsing it with the file format's `SKIP_HEADER=1` so the header row isn't loaded as data.
>
> </details>

---
## 4 · Book–Coffee Pairings (Labs 06, 04 & 05 — Marketplace + Cost + AI)

The café is launching a **reading corner**. The VP wants suggestions for which Alpine Peaks book categories pair best with Roastery drinks — like wine-and-food pairings, but for coffee and books.

> 📌 **The Marketplace dataset `ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES` is already here** — it was provisioned as part of Lab 06. You don't need to go back and run that lab. Just query it directly.

**Your task:**
1. Explore the Marketplace data — what book categories exist?
2. **Cost check:** estimate tokens with `AI_COUNT_TOKENS` on a sample pairing prompt. Is it cheap enough to run for every combination?
3. Use `AI_COMPLETE` to generate a creative pairing reason for each book-category × drink combination

> 📌 **Recall Lab 06:** Marketplace data is queried like any shared database — just reference it directly. **Recall Lab 04:** `AI_COUNT_TOKENS` tells you cost *before* you spend. Always estimate before scaling.

> 🤖 **CoCo assist** *(try the empty cells yourself first — if stuck, reveal the 🧩 solution below OR use this prompt):*
>
> *"Using the book categories in `ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES` and the beverages in `PLATPLUS_DB.ROASTERY_SALES.PRODUCTS` (where category = 'Beverage'), use AI_COMPLETE to suggest why each book category and coffee drink pair well together. Cross-join the two as CTEs and pass a prompt to SNOWFLAKE.CORTEX.AI_COMPLETE. Just generate the SQL — this is a starting point, so iterate with CoCo if the first result hardcodes values instead of reading the tables."*

In [ ]:
-- Explore the Alpine Peaks book categories


In [ ]:
-- Cost check: how many tokens would one pairing prompt use?
-- Syntax: AI_COUNT_TOKENS('<function_name>', '<model>', '<prompt>')


In [ ]:
-- Generate book-coffee pairings with AI_COMPLETE
-- Cross-join book categories with beverage names and ask the model why they pair well


> <details><summary>🧩 Stuck? Reference solution</summary>
>
> **Paste into `books_explore` cell:**
>
> ```sql
> -- Explore the Alpine Peaks book categories
> SELECT DISTINCT CATEGORY
> FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
> ORDER BY CATEGORY;
> ```
>
> **Paste into `token_check` cell:**
>
> ```sql
> -- Cost check: how many tokens would one pairing prompt use?
> SELECT AI_COUNT_TOKENS(
>   'ai_complete',
>   'llama3.3-70b',
>   'Suggest why a book about Cooking pairs well with a Latte at a coffee shop reading corner. One sentence.'
> ) AS estimated_tokens;
> -- ~34 tokens per prompt. 8 categories x 4 beverages = 32 calls. Very cheap.
> ```
>
> **Paste into `pairings` cell:**
>
> ```sql
> -- Generate book-coffee pairings with AI_COMPLETE
> WITH book_cats AS (
>   SELECT DISTINCT CATEGORY FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES
> ),
> drinks AS (
>   SELECT product_name FROM PLATPLUS_DB.ROASTERY_SALES.PRODUCTS WHERE category = 'Beverage'
> )
> SELECT b.CATEGORY AS book_category,
>        d.product_name AS drink,
>        SNOWFLAKE.CORTEX.AI_COMPLETE('llama3.3-70b',
>          'In one sentence, suggest why someone reading a ' || b.CATEGORY || ' book would enjoy a ' || d.product_name || ' at a coffee shop reading corner.'
>        ) AS pairing_reason
> FROM book_cats b
> CROSS JOIN drinks d
> ORDER BY b.CATEGORY, d.product_name;
> ```
>
> `AI_COUNT_TOKENS` takes the function name, the model, and the prompt — at ~34 tokens each across 32 calls this is trivially cheap, which is exactly why you check before scaling.
>
> </details>

---
## 5 · Feedback Intelligence (Labs 05 & 09 — AI Functions + Unstructured)

Now dig into *why* Mission is underperforming. Customer feedback holds the clues — but you need AI to read it at scale. Then cross-reference against the coffee knowledge-base PDFs on the stage to see if prep instructions could explain the complaints.

**Your task:**
1. Run `AI_SENTIMENT` on Roastery Mission's feedback (`store_id = 3`) to grade the sentiment of each comment
2. Run `AI_CLASSIFY` to bucket complaints into `['slow service', 'drink quality', 'pricing', 'ambiance', 'staff', 'other']`
3. Parse a coffee PDF from `@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE/coffee_kb/` with `AI_PARSE_DOCUMENT` (`'LAYOUT'` mode), then ask `AI_COMPLETE` whether the prep steps suggest what might go wrong (this one is filled in as your anchor)

> 📌 **Recall Lab 05:** `AI_SENTIMENT` grades a comment's sentiment (returns a category object — positive / neutral / negative, not a number). `AI_CLASSIFY` needs a list of candidate labels. **Recall Lab 09:** `TO_FILE('@stage', 'path')` + `AI_PARSE_DOCUMENT` turns a staged PDF into queryable text.

> 🤖 **CoCo assist** *(try the empty cells yourself first — if stuck, reveal the 🧩 solution below OR use this prompt):*
>
> *"Show me how to use AI_PARSE_DOCUMENT on a PDF at @PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE/coffee_kb/latte.pdf, then pass the parsed text into AI_COMPLETE to answer a question about it."*

In [ ]:
-- Grade the sentiment of each of Mission's feedback comments with AI_SENTIMENT (store_id = 3)


In [ ]:
-- Classify Mission's complaints into categories with AI_CLASSIFY


In [ ]:
-- Parse a coffee PDF and ask AI whether prep instructions match the complaints
WITH parsed AS (
  SELECT AI_PARSE_DOCUMENT(
    TO_FILE('@PLATPLUS_DB.ROASTERY_RAW.DOCS_STAGE', 'coffee_kb/latte.pdf'),
    {'mode': 'LAYOUT'}
  ):content::STRING AS doc_text
)
SELECT SNOWFLAKE.CORTEX.AI_COMPLETE('llama3.3-70b',
  'Based on this latte preparation guide: ' || doc_text ||
  ' --- If a customer complained about drink quality, does anything in these prep instructions suggest what could go wrong when staff skip or rush a step? Answer in 2-3 sentences.'
) AS ai_analysis
FROM parsed;


> <details><summary>🧩 Stuck? Reference solution</summary>
>
> **Paste into `feedback_sentiment` cell:**
>
> ```sql
> -- Grade the sentiment of each of Mission's feedback comments with AI_SENTIMENT (store_id = 3)
> SELECT feedback_id, comment,
>        SNOWFLAKE.CORTEX.AI_SENTIMENT(comment) AS sentiment_grade
> FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
> WHERE store_id = 3   -- Roastery Mission
> LIMIT 15;
> ```
>
> **Paste into `feedback_classify` cell:**
>
> ```sql
> -- Classify Mission's *complaints* (non-positive comments only) with AI_CLASSIFY
> WITH graded AS (
>   SELECT feedback_id, comment,
>          AI_CLASSIFY(comment,
>            ['slow service', 'drink quality', 'pricing', 'ambiance', 'staff', 'other']
>          ) AS complaint_category,
>          SNOWFLAKE.CORTEX.AI_SENTIMENT(comment):categories[0].sentiment::STRING AS sentiment_grade
>   FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
>   WHERE store_id = 3
> )
> SELECT feedback_id, comment, complaint_category
> FROM graded
> WHERE sentiment_grade != 'positive'
> LIMIT 15;
> ```
>
> The `doc_cross_ref` cell is already filled in — just run it. `AI_SENTIMENT` returns a category object per comment; `AI_CLASSIFY` needs the candidate-label list as its second argument. The PDF cross-reference shows how `AI_PARSE_DOCUMENT` turns an unstructured file into text you can feed straight into `AI_COMPLETE`.
>
> </details>

---
## 6 · Oops — Recovery (Lab 08 — Time Travel)

You're about to build your final deliverable. But first — an accident happens. You'll create an important temp table, then watch a bad `DELETE` wipe it. Real life: this happens more often than anyone admits.

**Your task:**
1. Run the `create_temp` cell — verify it has 6 rows
2. Run the `accident` cell — it deletes every row and captures the query ID
3. Use Time Travel to restore the rows: `INSERT INTO ... SELECT * FROM ... BEFORE(STATEMENT => $bad_query_id)`
4. Verify the table is restored

> 📌 **Recall Lab 08:** After a `DELETE`, the data isn't truly gone — Time Travel lets you query the table *as it was before* any statement. The syntax `BEFORE(STATEMENT => '<query_id>')` shows the table just before that query ran.

> 🤖 **CoCo assist** *(try the empty cells yourself first — if stuck, reveal the 🧩 solution below OR use this prompt):*
>
> *"I accidentally deleted all rows from my CAPSTONE_TEMP table. The query ID of the DELETE is stored in $bad_query_id. How do I use Time Travel to restore the rows?"*

In [ ]:
-- First, create a useful temp table (you'll need this later)
CREATE OR REPLACE TABLE {{my_db}}.SANDBOX.CAPSTONE_TEMP AS
SELECT store_name, 'this is important data' AS notes
FROM PLATPLUS_DB.ROASTERY_SALES.STORES;

-- Verify it exists (6 stores)
SELECT * FROM {{my_db}}.SANDBOX.CAPSTONE_TEMP;


> ⚠️ **Now run the next cell — it simulates an accident.** A bad `DELETE` removes every row from your table and captures its query ID. This is the moment Time Travel exists for.

In [ ]:
-- THE ACCIDENT: oops — every row just got deleted!
DELETE FROM {{my_db}}.SANDBOX.CAPSTONE_TEMP WHERE TRUE;

-- Capture the query ID of the bad statement NOW (before you run anything else)
SET bad_query_id = LAST_QUERY_ID();

-- Confirm it's empty
SELECT COUNT(*) AS rows_remaining FROM {{my_db}}.SANDBOX.CAPSTONE_TEMP;


In [ ]:
-- Recover! Use Time Travel to restore the deleted rows.
-- The query ID of the bad DELETE is stored in $bad_query_id


> <details><summary>🧩 Stuck? Reference solution</summary>
>
> **Paste into `recovery` cell:**
>
> ```sql
> -- Recover! Use Time Travel to restore the deleted rows.
> INSERT INTO {{my_db}}.SANDBOX.CAPSTONE_TEMP
> SELECT * FROM {{my_db}}.SANDBOX.CAPSTONE_TEMP BEFORE(STATEMENT => $bad_query_id);
>
> -- Verify recovery — should show 6 rows with store names
> SELECT * FROM {{my_db}}.SANDBOX.CAPSTONE_TEMP;
> ```
>
> **Why this works:** `BEFORE(STATEMENT => $bad_query_id)` queries the table as it existed *just before* the DELETE ran. The data is still in Snowflake's Time Travel storage — you're reading the historical version and inserting it back.
>
> </details>

---
## 7 · Final Deliverable — `V_ROASTERY_DIP` (Synthesis)

Package everything into one reusable view the ops team can re-run monthly without bugging you. This is the whole point: **a view is a deliverable, not a screenshot.**

**Your task:** create `{{my_db}}.SANDBOX.V_ROASTERY_DIP` combining:
- Store name
- Trailing 30-day vs prior 30-day revenue + `pct_change`
- Average sentiment score per store (from `PLATPLUS_DB.ROASTERY_SALES.FEEDBACK`) — use `SNOWFLAKE.CORTEX.SENTIMENT()` here, **not** `AI_SENTIMENT`: `SENTIMENT` returns a numeric score (−1 to 1) you can `AVG()`, whereas `AI_SENTIMENT` returns a category object you can't average
- Promo status per store (from your loaded `PROMOTIONS` table)

Then visualize it with a Plotly bar chart (the query + chart cells below are provided once your view exists).

> 📌 No single lab taught you to combine all three signals. This is where you prove you can compose independently.

> 🤖 **CoCo assist** *(try the empty cells yourself first — if stuck, reveal the 🧩 solution below OR use this prompt):*
>
> *"Using `PLATPLUS_DB.ROASTERY_SALES.ORDERS` + `STORES` for revenue, `PLATPLUS_DB.ROASTERY_SALES.FEEDBACK` for sentiment, and my `{{my_db}}.SANDBOX.PROMOTIONS` table for promo status, help me create the view `{{my_db}}.SANDBOX.V_ROASTERY_DIP` with three CTEs — revenue (last 30 days vs prior 30 days anchored to the most recent order date, with pct_change), sentiment (avg SNOWFLAKE.CORTEX.SENTIMENT per store), and promo status — joined on store. This is a starting point: iterate with CoCo until the view returns all four items from 'Your task' above."*

In [ ]:
-- Build your final view: V_ROASTERY_DIP
-- Combine: revenue window + average sentiment + promo status


In [ ]:
-- Query your view to verify it works
SELECT * FROM {{my_db}}.SANDBOX.V_ROASTERY_DIP ORDER BY PCT_CHANGE;


In [ ]:
# Final Plotly chart from V_ROASTERY_DIP
import plotly.express as px
from snowflake.snowpark.context import get_active_session
session = get_active_session()

df = session.sql(f"SELECT * FROM {my_db}.SANDBOX.V_ROASTERY_DIP").to_pandas()
df = df.sort_values('PCT_CHANGE')
max_ts = session.sql("SELECT MAX(order_ts)::DATE FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS").collect()[0][0]
fig = px.bar(df, x='STORE_NAME', y='PCT_CHANGE',
             title=f'Roastery Intelligence — Revenue % Change by Store as of {max_ts} (with Sentiment & Promo Context)',
             color_discrete_sequence=['#29B5E8'])
fig.update_layout(xaxis_title="Store", yaxis_title="% change")
fig


> <details><summary>🧩 Stuck? Reference solution</summary>
>
> **Paste into `final_view` cell:**
>
> ```sql
> -- Build your final view: V_ROASTERY_DIP
> CREATE OR REPLACE VIEW {{my_db}}.SANDBOX.V_ROASTERY_DIP AS
> WITH latest AS (
>   SELECT MAX(order_ts) AS max_ts FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS
> ),
> revenue AS (
>   SELECT s.store_id, s.store_name,
>     SUM(CASE WHEN o.order_ts >= DATEADD(day,-30,l.max_ts) THEN o.total_amount ELSE 0 END) AS sales_last_30d,
>     SUM(CASE WHEN o.order_ts >= DATEADD(day,-60,l.max_ts)
>               AND o.order_ts < DATEADD(day,-30,l.max_ts) THEN o.total_amount ELSE 0 END) AS sales_prior_30d,
>     ROUND(100.0 * (sales_last_30d - sales_prior_30d) / NULLIF(sales_prior_30d, 0), 1) AS pct_change
>   FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS o
>   JOIN PLATPLUS_DB.ROASTERY_SALES.STORES s ON s.store_id = o.store_id
>   CROSS JOIN latest l
>   WHERE o.order_ts >= DATEADD(day,-60, l.max_ts)
>   GROUP BY s.store_id, s.store_name
> ),
> sentiment AS (
>   SELECT store_id,
>          ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(comment)), 3) AS avg_sentiment
>   FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK
>   GROUP BY store_id
> ),
> promos AS (
>   SELECT store_name,
>          MAX(CASE WHEN status = 'cancelled' THEN 'cancelled' ELSE 'ran' END) AS promo_status
>   FROM {{my_db}}.SANDBOX.PROMOTIONS
>   GROUP BY store_name
> )
> SELECT r.store_name, r.sales_last_30d, r.sales_prior_30d, r.pct_change,
>        s.avg_sentiment, p.promo_status
> FROM revenue r
> LEFT JOIN sentiment s ON s.store_id = r.store_id
> LEFT JOIN promos p ON p.store_name = r.store_name;
> ```
>
> **Note:** (`SNOWFLAKE.CORTEX.SENTIMENT()` is used here — not `AI_SENTIMENT` — because it returns an averageable numeric score; see the task above.) Read the three signals *together* and they converge: **Mission** has the **steepest revenue drop** (`pct_change` ≈ −64%), the **lowest `avg_sentiment`** (≈ 0.33, a clear outlier), **and** a **`cancelled` promo**. Three independent signals — revenue, sentiment, and promotions — all point at the same store. That's the payoff of composing them into one view: the dip isn't a fluke, it's corroborated from three directions.
>
> </details>

---
## Check Your Work
Run the next cell to **check your work**. It queries your own database to confirm you created the objects this capstone asked for and prints a ✅ / ❌ for each — no need to read the code.

In [ ]:
# --- Automated verification ---
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'

checks = []

# Check 1: V_ROASTERY_DIP exists and has rows
try:
    dip_rows = session.sql(f"SELECT COUNT(*) AS c FROM {my_db}.SANDBOX.V_ROASTERY_DIP").collect()[0]['C']
    checks.append(('V_ROASTERY_DIP exists', dip_rows >= 1, f'{dip_rows} rows'))
except Exception as e:
    checks.append(('V_ROASTERY_DIP exists', False, str(e)[:80]))

# Check 2: PROMOTIONS table loaded
try:
    promo_rows = session.sql(f"SELECT COUNT(*) AS c FROM {my_db}.SANDBOX.PROMOTIONS").collect()[0]['C']
    checks.append(('PROMOTIONS table loaded', promo_rows >= 10, f'{promo_rows} rows'))
except Exception as e:
    checks.append(('PROMOTIONS table loaded', False, str(e)[:80]))

# Check 3: CAPSTONE_TEMP recovered (has its columns back)
try:
    cols = session.sql(f"SELECT * FROM {my_db}.SANDBOX.CAPSTONE_TEMP LIMIT 1").columns
    checks.append(('CAPSTONE_TEMP recovered', len(cols) >= 2, f'{len(cols)} columns'))
except Exception as e:
    checks.append(('CAPSTONE_TEMP recovered', False, str(e)[:80]))

# Check 4: stale products excluded from V_PRODUCTS_FRESH (verify-before-trust twist)
try:
    stale = session.sql(f"SELECT COUNT(*) AS c FROM {my_db}.SANDBOX.V_PRODUCTS_FRESH WHERE product_id IN (103, 106)").collect()[0]['C']
    checks.append(('Stale products excluded (V_PRODUCTS_FRESH)', stale == 0, f'{stale} still present'))
except Exception as e:
    checks.append(('Stale products excluded (V_PRODUCTS_FRESH)', False, str(e)[:80]))

print('━' * 60)
print('CAPSTONE VERIFICATION')
print('━' * 60)
all_pass = True
for name, passed, detail in checks:
    status = '✅' if passed else '❌'
    print(f"  {status} {name:<42} ({detail})")
    if not passed:
        all_pass = False
print('━' * 60)
if all_pass:
    print('\n🎓 All checks passed — capstone complete!')
else:
    print('\n⚠️  Some checks failed — review the sections above, re-run, then run this cell again.')

> 📌 **Note:** This only checks sections that create persistent objects (tables, views). Sections that produce one-off query results (RBAC diagnostics, charts, AI function output, book pairings) aren't checked here — you verified those by reading their output as you went.

---
### :compass: Open reflection

Before you wrap up, answer these for yourself (there's no single right answer):

1. **What story do the three signals tell together?** Revenue dip + cancelled promos + negative sentiment — is there a single root cause, or several contributing factors?
2. **Which lab skill was hardest to recall?** That's your signal for what to review before using Snowflake for real.
3. **What would you add to `V_ROASTERY_DIP` if you had more time?** The book pairings? Complaint categories? A freshness-check column?
4. **The RBAC moment at the start** — in a real org, would you have known what to do if you hit "does not exist or not authorized" without Lab 03?
5. **CoCo usage:** did you ask CoCo for help? That's not cheating — it's how analysts work with AI assistants in production. The skill is *knowing what to ask*.

### ❄️ Key Takeaways

- **RBAC first:** verify your role has access before starting an investigation
- **Verify before you trust:** check a handoff's freshness column — a stale row hiding in a fresh table is easy to miss
- **Cost before compute:** `AI_COUNT_TOKENS` before `AI_COMPLETE` at scale — small habit, big savings
- **Load, don't assume:** the promo CSV revealed Mission's cancelled promotions — data that lived outside Snowflake until you brought it in
- **AI as an analytical partner:** sentiment scores quantify gut feelings; classification buckets make feedback actionable
- **Accidents happen:** Time Travel turns a disaster into a 30-second recovery — but only if you know it exists
- **Package your work:** a reusable view is a deliverable; a one-off query is a screenshot that expires

---

**Labs recalled in this capstone:**

| Lab | Skill | Where it appeared |
|-----|-------|-------------------|
| 01 | Navigate objects & joins | Section 2 — revenue query |
| 02 | Verify results | Section 2 — rolling window + freshness check |
| 03 | RBAC & access | Section 1 — access gate |
| 04 | Credits & cost | Section 4 — AI_COUNT_TOKENS |
| 05 | AI functions | Sections 4 & 5 — sentiment, classify, complete |
| 06 | Marketplace | Section 4 — Alpine Peaks books |
| 07 | Getting data in | Section 3 — stage + COPY INTO |
| 08 | Time Travel | Section 6 — recovery |
| 09 | Unstructured AI | Section 5 — parse PDF + cross-reference |

### Notebook completion. 🏁
You're done — and you've just run an investigation that touched every skill in the course. Your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.